In [2]:
# Cell 1 — Import libraries and load cleaned data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load the original CSV again (fresh start for ML notebook)
df = pd.read_csv(r"D:\Customer Lifetime Value & Churn Predictor\Data\WA_Fn-UseC_-Telco-Customer-Churn.csv")
# Repeat the 3 cleaning steps
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("✅ Data loaded and cleaned")
print("Shape:", df.shape)

✅ Data loaded and cleaned
Shape: (7043, 21)


In [3]:
# Cell 2 — Prepare features and target

# Step 1 — Drop columns we don't need
df_ml = df.drop(['customerID', 'TotalCharges'], axis=1)

# Step 2 — Encode target column (Churn Yes=1, No=0)
df_ml['Churn'] = df_ml['Churn'].map({'Yes': 1, 'No': 0})

# Step 3 — Encode all remaining text columns
df_ml = pd.get_dummies(df_ml, drop_first=True)

# Step 4 — Separate features and target
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

print("✅ Data prepared for ML")
print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
print(X.columns.tolist())

✅ Data prepared for ML
Features shape : (7043, 29)
Target shape   : (7043,)

Feature columns (29):
['tenure', 'MonthlyCharges', 'gender_Male', 'SeniorCitizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [4]:
# Cell 3 — Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print("✅ Data split complete")
print(f"Training set   : {X_train.shape[0]} customers (80%)")
print(f"Testing set    : {X_test.shape[0]} customers (20%)")
print(f"\nChurn rate in training set: {y_train.mean()*100:.2f}%")
print(f"Churn rate in testing set : {y_test.mean()*100:.2f}%")

✅ Data split complete
Training set   : 5634 customers (80%)
Testing set    : 1409 customers (20%)

Churn rate in training set: 26.55%
Churn rate in testing set : 26.47%


In [5]:
# Cell 4 — Train the Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("✅ Model trained successfully")
print(f"Algorithm      : Random Forest")
print(f"Trees built    : 100")
print(f"Trained on     : {X_train.shape[0]} customers")
print(f"Features used  : {X_train.shape[1]} features")

✅ Model trained successfully
Algorithm      : Random Forest
Trees built    : 100
Trained on     : 5634 customers
Features used  : 29 features


In [6]:
# Cell 5 — Test the model and measure accuracy

# Step 1 — Make predictions on test data
y_pred = rf_model.predict(X_test)

# Step 2 — Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print("=== Model Performance ===")
print(f"Accuracy Score : {accuracy*100:.2f}%")

# Step 3 — Detailed report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

# Step 4 — Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== Model Performance ===
Accuracy Score : 81.05%

Detailed Classification Report:
              precision    recall  f1-score   support

      Stayed       0.84      0.92      0.88      1036
     Churned       0.69      0.51      0.59       373

    accuracy                           0.81      1409
   macro avg       0.77      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix:
[[952  84]
 [183 190]]


In [7]:
# Cell 6 — Feature Importance

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
})

feature_importance = feature_importance\
    .sort_values('importance', ascending=False)\
    .reset_index(drop=True)

print("=== Top 15 Most Important Features ===")
print(feature_importance.head(15))


=== Top 15 Most Important Features ===
                           feature  importance
0                           tenure    0.246950
1                   MonthlyCharges    0.136571
2                Contract_Two year    0.080054
3      InternetService_Fiber optic    0.071665
4   PaymentMethod_Electronic check    0.062456
5                Contract_One year    0.046130
6               OnlineSecurity_Yes    0.038935
7                  TechSupport_Yes    0.032826
8             PaperlessBilling_Yes    0.027546
9                      Partner_Yes    0.020543
10                OnlineBackup_Yes    0.020025
11                     gender_Male    0.018622
12                  Dependents_Yes    0.018511
13               MultipleLines_Yes    0.016359
14            DeviceProtection_Yes    0.016322
